In [ ]:
import findspark
findspark.init("/usr/local/spark")

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("ML").getOrCreate()

In [ ]:
df = spark.read.csv("churn.csv",inferSchema=True, header=True)

In [ ]:
# Explore, clean, analyse & visualise the data

In [ ]:
df1=df.toPandas()

In [ ]:
df1.shape

In [ ]:
df1.info()

In [ ]:
df1.isna().sum()


In [ ]:
round(df1.describe(),2).T

In [ ]:
# Look for class distribution of the target variable

In [ ]:
df1['Exited'].value_counts()

In [ ]:
# Baseline accuracy
7963/10000

In [ ]:
df1[:5]

In [ ]:
df1['Geography'].value_counts()

In [ ]:
df1.groupby('Geography').Exited.mean()

In [ ]:
import numpy as np

In [ ]:
df1.groupby('Geography').agg({'Exited':np.mean,'EstimatedSalary':['min','max',np.mean],
                             'Balance':[np.mean,'min','max']})

In [ ]:
df1['Geography'][:5]

In [ ]:
import pandas as pd

In [ ]:
pd.get_dummies(df1['Geography'], drop_first=True)[:5]

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [ ]:
cat_id = StringIndexer(inputCol= "Geography", outputCol='Geo').fit(df)

In [ ]:
df = cat_id.transform(df)

In [ ]:
df.select('Geo','Geography').show(5)

In [ ]:
cat_id1 = StringIndexer(inputCol= "Gender", outputCol='Gender1').fit(df)

In [ ]:
df = cat_id1.transform(df)

In [ ]:
df.select('Gender','Gender1').show(5)

In [ ]:
cat_one = OneHotEncoder(inputCol="Geo", outputCol="Geo_encoded")

In [ ]:
df=cat_one.transform(df)

In [ ]:
df.select('Geo','Geo_encoded').show()

In [ ]:
cat_two = OneHotEncoder(inputCol="Gender1", outputCol="Gen_encoded")

In [ ]:
df=cat_two.transform(df)

In [ ]:
df.columns

In [ ]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

In [ ]:
df_assembler = VectorAssembler(inputCols=['CreditScore','Age','Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Geo_encoded',
 'Gen_encoded'], outputCol="features")

In [ ]:
df = df_assembler.transform(df)

In [ ]:
df.select('features').show(5)

In [ ]:
final_df=df.select("features","Exited")

In [ ]:
final_df.show(5)

In [ ]:
train_data,test_data = final_df.randomSplit([0.8,0.2])

In [ ]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier

In [ ]:
lr = LogisticRegression(labelCol='Exited')

In [ ]:
model = lr.fit(train_data)

In [ ]:
pred_churn=model.evaluate(test_data)

In [ ]:
pred_churn.predictions.show(15)

In [ ]:
type(pred_churn)

In [ ]:
pred_churn.predictions.collect()

In [54]:
pred = model.transform(test_data)

In [55]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [56]:
eval = BinaryClassificationEvaluator(rawPredictionCol='prediction',labelCol='Exited')

In [57]:
eval.evaluate(pred)

0.5842732802910572

In [ ]:
help(BinaryClassificationEvaluator)

In [58]:
dtree = DecisionTreeClassifier(featuresCol='features', labelCol='Exited')

In [59]:
model_tree = dtree.fit(train_data)

In [60]:
pred_tree=model_tree.transform(test_data)

In [61]:
pred_tree

DataFrame[features: vector, Exited: int, rawPrediction: vector, probability: vector, prediction: double]

In [62]:
pred_tree.show()

+--------------------+------+--------------+--------------------+----------+
|            features|Exited| rawPrediction|         probability|prediction|
+--------------------+------+--------------+--------------------+----------+
|(11,[0,1,2,3,4,7]...|     1| [129.0,131.0]|[0.49615384615384...|       1.0|
|(11,[0,1,2,3,4,7]...|     1|[1362.0,147.0]|[0.90258449304174...|       0.0|
|(11,[0,1,2,3,4,7]...|     0| [1095.0,92.0]|[0.92249368155012...|       0.0|
|(11,[0,1,2,3,4,7]...|     0| [129.0,131.0]|[0.49615384615384...|       1.0|
|(11,[0,1,2,3,4,7]...|     0|  [190.0,46.0]|[0.80508474576271...|       0.0|
|(11,[0,1,2,3,4,7]...|     0|[1362.0,147.0]|[0.90258449304174...|       0.0|
|(11,[0,1,2,3,4,7]...|     0|[1362.0,147.0]|[0.90258449304174...|       0.0|
|(11,[0,1,2,3,4,7]...|     0| [1095.0,92.0]|[0.92249368155012...|       0.0|
|(11,[0,1,2,3,4,7]...|     1|[1362.0,147.0]|[0.90258449304174...|       0.0|
|(11,[0,1,2,3,4,7]...|     0|[1362.0,147.0]|[0.90258449304174...|       0.0|

In [63]:
eval_tree = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='Exited')

In [64]:
acc = eval_tree.evaluate(pred_tree)

In [65]:
acc

0.8431652398400369

In [66]:
from sklearn.metrics import confusion_matrix

In [3]:
y_origin = pred_tree.select('Exited').collect()

In [4]:
y_predicted = pred_tree.select('prediction').collect()

In [ ]:
pd.Series(y_origin)

In [6]:
# pd.crosstab(y_origin,y_predicted)

In [7]:
cm = confusion_matrix(y_true=y_origin,y_pred=y_predicted)

In [ ]:
197/(197+86)

In [ ]:
197/(201+197)